# Two faster correction methods

[Grid24-RSC](../src/rsc/methods/grid24.py) estimates motion on a 24-region grid and corrects the full frame with one warp. [DIS-RSC](../src/rsc/methods/dis_rsc.py) estimates per-pixel motion at quarter scale and corrects the full frame. Both use [weighted ridge regression](../src/rsc/training) to fit the corrections. [TMCD](https://github.com/evgobmm/TMCD) supplies robust training weights.

In [1]:
from csv import DictReader
from pathlib import Path
from statistics import mean
import os
import subprocess
import sys

from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "results" / "timings.csv").is_file())

def read_results(filename):
    with (ROOT / "results" / filename).open(encoding="utf-8", newline="") as file:
        return list(DictReader(file))

quality = read_results("quality.csv")
timings = read_results("timings.csv")

def average_quality(method):
    rows = [row for row in quality
            if row["method"] == method and 1 <= int(row["frame"]) <= 48]
    assert len(rows) == 720
    return mean(float(row["psnr_db"]) for row in rows), mean(float(row["ssim"]) for row in rows)

def latency(method):
    return mean(float(row["milliseconds"]) for row in timings if row["method"] == method)

def show_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separators = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join([header, separators, *body])))

## Compare with Uni-RIFE

Quality uses the same 720 frame pairs for all three methods. Correction time is the mean over 120 frames for Uni-RIFE and DIS-RSC, and four repeats of that schedule for Grid24-RSC. The clip estimate scales the per-frame time to one minute at 30 fps.

In [2]:
methods = [("uni-rife", "Uni-RIFE"), ("grid24-rsc", "Grid24-RSC"), ("dis-rsc", "DIS-RSC")]
baseline_ms = latency("uni-rife")
rows = []
for method, name in methods:
    ms = latency(method)
    psnr, ssim = average_quality(method)
    rows.append((name, f"{ms:.3f}", f"{baseline_ms / ms:,.1f}×", f"{ms * 1.8:.2f}",
                 f"{psnr:.3f}", f"{ssim:.5f}"))

show_table(("Method", "Mean ms/frame", "Speedup", "Seconds per 1-min clip", "PSNR (dB)", "SSIM"), rows)

| Method | Mean ms/frame | Speedup | Seconds per 1-min clip | PSNR (dB) | SSIM |
| --- | --- | --- | --- | --- | --- |
| Uni-RIFE | 556.987 | 1.0× | 1002.58 | 31.586 | 0.92588 |
| Grid24-RSC | 0.322 | 1,729.6× | 0.58 | 30.887 | 0.91525 |
| DIS-RSC | 38.834 | 14.3× | 69.90 | 31.919 | 0.92726 |

## Rerun locally

Follow the [setup instructions](../REPRODUCE.md#run-on-windows), then set the run size below.

In [3]:
run = "saved"  # Change to "quick" for three frames, or "full" for all test scenes.
device = "recorded"  # "cpu" runs all methods on the CPU.
data = Path("data/BS-RSC")
if run in ("quick", "full"):
    command = [
        sys.executable, "-u", "-m", "rsc.benchmark", "--suite", "ours",
        "--" + run, "--device", device, "--data", str(data),
        "--output", "runs/notebook-ours",
    ]
    with subprocess.Popen(
        command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        encoding="utf-8", errors="replace",
        env={**os.environ, "OPENCV_OPENCL_BUILD_EXTRA_OPTIONS": "-cl-std=CL2.0"},
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
else:
    print("Showing the saved report run.")

Showing the saved report run.
